# Train a loan default model and save it for a Streamlit app

Run this notebook in Google Colab. It trains a simple logistic regression classifier and creates the two files your Streamlit app needs:

- `model.pkl`: the trained model, saved with the pickle package
- `requirements.txt`: the packages (and versions) the app needs

The app itself is in a separate file, `app.py`.

### Import needed packages

In [1]:
import pandas as pd
import numpy as np
import sklearn
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

### Read in the data
The file is read straight from the GitHub repository, so nothing needs to be uploaded to Colab.

In [2]:
url = 'https://raw.githubusercontent.com/MattDBailey/ANOP330Module_LoanDefaultPredictor-Streamlit/main/LendingClub_DataSet.csv'
lend_df = pd.read_csv(url)
lend_df.head(10)

,home_ownership,income,dti,fico,loan_status
0,1,44304.0,18.47,690,0
1,0,50000.0,29.62,735,1
2,0,64400.0,16.68,675,1
3,0,38500.0,33.73,660,0
4,1,118000.0,26.66,665,1
5,1,43000.0,20.68,725,1
6,0,76000.0,17.31,685,1
7,0,75000.0,22.34,700,1
8,1,156000.0,13.28,735,1
9,1,54000.0,19.00,660,0


### Create the target
In this file `loan_status` is 1 when the loan was **repaid** and 0 when it **defaulted**.

The event we want to predict is default, so we make default the positive class (1). That way `predict_proba(...)[:, 1]` is the probability of default.

In [3]:
lend_df['default'] = 1 - lend_df['loan_status']

# Share of loans that defaulted (1) and were repaid (0)
lend_df['default'].value_counts(normalize=True)

,proportion
default,
0,0.791945
1,0.208055


In [4]:
# Separate into predictors (X) and a SINGLE TARGET outcome (y)
X = lend_df[['home_ownership', 'income', 'dti', 'fico']]
y = lend_df['default']

In [5]:
# Create a train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, train_size=.8)

### Use GridSearchCV with 5 folds to find a decent logistic regression classifier

In [8]:
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

gridSearch_auc = GridSearchCV(estimator=LogisticRegression(solver='liblinear', random_state=1),
                              param_grid=param_grid, cv=5, scoring='roc_auc', return_train_score=True)
gridSearch_auc.fit(X_train, y_train)

# Get index of the best parameters
best_index = gridSearch_auc.best_index_
mean_train_score = gridSearch_auc.cv_results_['mean_train_score'][best_index]
mean_test_score = gridSearch_auc.cv_results_['mean_test_score'][best_index]

# print the mean score from GridSearchCV and the CV score
print('Mean training AUC for the best C: {:.2f}'.format(mean_train_score))
print('Cross-validated AUC: {:.2f}'.format(mean_test_score))
print('Best parameters from param_grid:', gridSearch_auc.best_params_)

# Keep the model with the best parameters
BestLogReg = gridSearch_auc.best_estimator_

Mean training AUC for the best C: 0.63
Cross-validated AUC: 0.63
Best parameters from param_grid: {'C': 0.001}


### The model isn't great, but we're building a simple model to illustrate the principle. We still check its performance on the test set.

In [9]:
# Column 1 of predict_proba is the probability of default
test_pred_prob = BestLogReg.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_pred_prob)
print('Test AUC: {:.2f}'.format(test_auc))

Test AUC: 0.61


### Save the model as a file
Now you can use the model elsewhere without rebuilding it or sharing the training data. `model.pkl` appears in the Colab Files panel (the folder icon on the left).

In [10]:
# Write the model to a binary file ('wb') using pickle
filename = 'model.pkl'
with open(filename, 'wb') as file:
    pickle.dump(BestLogReg, file)

### Check the saved model: two outputs for one new customer
This is the same code the Streamlit app uses. The model returns two things:

1. the **predicted class** from `.predict()` (1 = default, 0 = not default)
2. the **predicted probability of default** from `.predict_proba()`

In [11]:
# Open the file ('rb') and load the model
file_to_load = 'model.pkl'
with open(file_to_load, 'rb') as file:
    loaded_model = pickle.load(file)

# Create a new customer
ownhome = 1     # 1 for own home, 0 for rent
income = 80000  # annual income in dollars
dti = 20        # debt-to-income ratio in percent (20 means 20%)
fico = 700      # FICO score

# The column names and their ORDER must match the training data
new_customer = pd.DataFrame({
    'home_ownership': [ownhome],
    'income': [income],
    'dti': [dti],
    'fico': [fico]
})

# Output 1: predicted class
predicted_class = loaded_model.predict(new_customer)[0]
# Output 2: predicted probability of default
predicted_prob = loaded_model.predict_proba(new_customer)[0, 1]

print('Predicted probability of default: {:.2f}'.format(predicted_prob))
if predicted_class == 1:
    print('Predicted class: Default')
else:
    print('Predicted class: Not Default')

Predicted probability of default: 0.21
Predicted class: Not Default


### Create requirements.txt
The app must load the model with the **same package versions** that created it. This cell writes the versions Colab is using into `requirements.txt`.

In [12]:
with open('requirements.txt', 'w') as file:
    file.write('streamlit\n')
    file.write('pandas==' + pd.__version__ + '\n')
    file.write('numpy==' + np.__version__ + '\n')
    file.write('scikit-learn==' + sklearn.__version__ + '\n')

print(open('requirements.txt').read())

streamlit
pandas==2.2.3
numpy==2.4.6
scikit-learn==1.6.1



### Download the two files
In the Colab Files panel (folder icon on the left), right-click `model.pkl` and choose **Download**. Do the same for `requirements.txt`.

Upload both to your GitHub repository alongside `app.py`.

**You do not need to upload your training data.** Everything the model learned is stored in `model.pkl`, so the app needs only that file. When you build your own model with your own data, the data stays in Colab.

**If your own model needs scaling:** put the scaler and the model together in a pipeline, as we did with `StandardScaler` inside `GridSearchCV`, and pickle the pipeline. Then the app can send raw inputs and the scaling happens automatically.

In [14]:
Python version that Streamlit may ask for when you run the app:
import sys; print(sys.version)

3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
